In [ ]:
import pandas as pd
from IPython.display import display

path = r"/data/yelp_academic_dataset_review.json"   # change if needed

df = pd.read_json(path, lines=True, nrows=5000)
cols = ["review_id", "user_id", "business_id", "stars", "date", "text"]

# shorten long text so the table stays readable
df["text_snip"] = df["text"].str.slice(0, 160)

display(df[["review_id","user_id","business_id","stars","date","text_snip"]].head(100))


,review_id,user_id,business_id,stars,date,text_snip
0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3,2018-07-07 22:09:11,"If you decide to eat here, just be aware it is..."
1,BiTunyQ73aT9WBnpR9DZGw,OyoGAe7OKpv6SyGZT5g77Q,7ATYjTIgM3jUlt4UM3IypQ,5,2012-01-03 15:28:18,I've taken a lot of spin classes over the year...
2,saUsX_uimxRlCVr67Z4Jig,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3,2014-02-05 20:30:30,Family diner. Had the buffet. Eclectic assortm...
3,AqPFMleE6RsU23_auESxiA,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5,2015-01-04 00:01:03,"Wow! Yummy, different, delicious. Our favo..."
4,Sx8TMOWLNuJBWer-0pcmoA,bcjbaE6dDog4jkNY91ncLQ,e4Vwtrqf-wpJfwesgvdgxQ,4,2017-01-14 20:54:15,Cute interior and owner (?) gave us tour of up...
...,...,...,...,...,...,...
95,QS7CuOtFLuS3dnwKHRtSYQ,PBnEwGVCBL0N-bET6ZI6kQ,m5-FtgWRd4qA7j0vaOXiXQ,5,2016-11-10 20:56:13,Had to wait until my third trip to NOLA to act...
96,4PHFo_GRG4FEk1q4X7xQVQ,jbsCBG0A-3wVDjrKar-0Wg,X63jIMRHYBvKKQDuJTRiQQ,5,2014-10-11 13:55:05,A GREAT EXPERIENCE!!!!!!!!! I was a completel...
97,1c6sgLe07HAhipebsQ1wgA,ZRXvbrutBBULaFS6T9NCwA,HnhxO2cpa15AHI1456Pl6Q,5,2015-10-17 00:55:35,Wow! I never thought my sons phone could be re...
98,PPgbLBvi34A6m7bKJfTwhw,3TL6HZ1JrKcNTvGDWKlrow,GyC36Pn0Q1-qHnqXys6yFg,1,2013-12-07 13:17:13,Service and management terrible... After messi...


In [ ]:
import pandas as pd
from pathlib import Path

path = "data/yelp_academic_dataset_review.json"   # adjust if needed
out_dir = Path("data/prepped"); out_dir.mkdir(parents=True, exist_ok=True)

use_cols = ["review_id","user_id","business_id","stars","date","text"]
df = pd.read_json(path, lines=True)[use_cols].dropna(subset=["user_id","business_id","date"])

# rename → event_time (ISO-8601) and trim whitespace
df = df.rename(columns={"date":"event_time"})
df["event_time"] = pd.to_datetime(df["event_time"]).dt.tz_localize("UTC").dt.strftime("%Y-%m-%dT%H:%M:%SZ")
df["text"] = df["text"].str.strip()

# make small + medium slices
df.head(3000).to_json(out_dir/"yelp_small.jsonl", orient="records", lines=True)
df.head(75000).to_json(out_dir/"yelp_medium.jsonl", orient="records", lines=True)

print("wrote:", (out_dir/"yelp_small.jsonl").as_posix(), (out_dir/"yelp_medium.jsonl").as_posix())


In [1]:
from pathlib import Path
import json, itertools

src = Path("../data/yelp_academic_dataset_review.json")   # adjust if your notebook is in /notebooks
dst = Path("../data/prepped/yelp_small.jsonl")
dst.parent.mkdir(parents=True, exist_ok=True)

need = ("review_id", "user_id", "business_id", "stars", "text", "date")

with src.open("r", encoding="utf-8") as fin, dst.open("w", encoding="utf-8") as fout:
    for line in itertools.islice(fin, 3000):           # <- only first 3k lines
        obj = json.loads(line)
        row = {k: obj.get(k) for k in need}
        # rename date -> event_time (keep as string for now)
        row["event_time"] = row.pop("date", None)
        fout.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Wrote:", dst)


Wrote: ../data/prepped/yelp_small.jsonl


In [1]:
import pandas as pd, glob
from pathlib import Path

# resolve project root no matter where the notebook runs
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

bronze_dir = root / "delta" / "bronze_reviews"
metric_dir = root / "delta" / "metrics_1m"

# list parquet files
bronze_paths = sorted(glob.glob(str(bronze_dir / "*.parquet")))
metric_paths = sorted(glob.glob(str(metric_dir / "*.parquet")))

# read one file (quick peek)
pd.read_parquet(bronze_paths[0]).head()

# or read all files
bronze = pd.concat((pd.read_parquet(p) for p in bronze_paths), ignore_index=True)
metrics = pd.concat((pd.read_parquet(p) for p in metric_paths), ignore_index=True)

len(bronze), metrics.sort_values("window_start").tail(5)


(30600,
           window_start          window_end  n_reviews  avg_stars
 40 2021-12-31 06:13:00 2021-12-31 06:14:00          1        5.0
 4  2022-01-12 08:26:00 2022-01-12 08:27:00          1        5.0
 34 2022-01-13 10:13:00 2022-01-13 10:14:00          1        2.0
 93 2022-01-13 10:24:00 2022-01-13 10:25:00          1        5.0
 33 2022-01-16 04:59:00 2022-01-16 05:00:00          1        5.0)

In [3]:
import glob, pandas as pd
paths = glob.glob("../delta/metrics_1m/*.parquet")
metrics = pd.concat([pd.read_parquet(p) for p in paths], ignore_index=True)
metrics.sort_values("window_start").tail(10)


,window_start,window_end,n_reviews,avg_stars
836,2018-09-26 03:38:00,2018-09-26 03:39:00,2,5.0
1128,2018-10-03 11:07:00,2018-10-03 11:08:00,2,1.0
2289,2018-10-03 21:55:00,2018-10-03 21:56:00,2,5.0
1178,2018-10-04 08:15:00,2018-10-04 08:16:00,2,1.0
198,2018-10-04 20:13:00,2018-10-04 20:14:00,2,5.0
906,2018-10-04 20:24:00,2018-10-04 20:25:00,2,5.0
1681,2018-10-04 23:18:00,2018-10-04 23:19:00,2,5.0
2405,2018-10-04 23:58:00,2018-10-04 23:59:00,2,5.0
845,2018-10-05 00:11:00,2018-10-05 00:12:00,2,2.0
946,2018-10-05 00:33:00,2018-10-05 00:34:00,2,3.0


In [4]:
import glob, pandas as pd
metrics = pd.concat([pd.read_parquet(p) for p in glob.glob("../delta/metrics_1m/*.parquet")],
                    ignore_index=True)
metrics.sort_values("window_start").tail(10)


,window_start,window_end,n_reviews,avg_stars
836,2018-09-26 03:38:00,2018-09-26 03:39:00,2,5.0
1128,2018-10-03 11:07:00,2018-10-03 11:08:00,2,1.0
2289,2018-10-03 21:55:00,2018-10-03 21:56:00,2,5.0
1178,2018-10-04 08:15:00,2018-10-04 08:16:00,2,1.0
198,2018-10-04 20:13:00,2018-10-04 20:14:00,2,5.0
906,2018-10-04 20:24:00,2018-10-04 20:25:00,2,5.0
1681,2018-10-04 23:18:00,2018-10-04 23:19:00,2,5.0
2405,2018-10-04 23:58:00,2018-10-04 23:59:00,2,5.0
845,2018-10-05 00:11:00,2018-10-05 00:12:00,2,2.0
946,2018-10-05 00:33:00,2018-10-05 00:34:00,2,3.0


In [5]:
import glob, pandas as pd
metrics = pd.concat([pd.read_parquet(p) for p in glob.glob("../delta/metrics_1m/*.parquet")], ignore_index=True)
metrics.sort_values("window_start").tail()


,window_start,window_end,n_reviews,avg_stars
906,2018-10-04 20:24:00,2018-10-04 20:25:00,2,5.0
1681,2018-10-04 23:18:00,2018-10-04 23:19:00,2,5.0
2405,2018-10-04 23:58:00,2018-10-04 23:59:00,2,5.0
845,2018-10-05 00:11:00,2018-10-05 00:12:00,2,2.0
946,2018-10-05 00:33:00,2018-10-05 00:34:00,2,3.0
